In [1]:
import geopandas as gpd
import pandas as pd
import os
import geopandas as gpd
import os
import pandas as pd
from pathlib import Path
# import linktransformer as lt
import numpy as np

main_dir =  os.path.dirname(os.path.dirname(os.getcwd()))


In [2]:
shell_path = ""
dbox_path = r"C:\Users\eunic\Dropbox\sa_fires"
root = dbox_path
input_path = fr"{root}\data\input"
int_path = fr"{root}\proj_bureaucrats_farms\data_output\intermediate"

In [3]:
# Define the base directory
base_dir = f"{root}/data/input/my_neta/winner_details_sel_st"

# Initialize a list to store dataframes
dfs = []

# Walk through all subdirectories
for subdir, _, files in os.walk(base_dir):
    if "info.csv" in files:  # Check if 'info.csv' exists in the folder
        file_path = os.path.join(subdir, "info.csv")  # Full path to the file
        subfolder_name = Path(subdir).name  # Extract subfolder name
        
        # Read CSV file
        df = pd.read_csv(file_path)
        
        # Add the subfolder name as a new column
        df["subfolder"] = subfolder_name
        
        # Append to the list
        dfs.append(df)

In [4]:
# Concatenate all dataframes
final_df = pd.concat(dfs, ignore_index=True)

# Extract year (4 digits after the letters)
final_df["year"] = final_df["subfolder"].str.extract(r'(\d{4})')
cols = ['constituency', 'year' ]
final_df[cols].duplicated().sum()

np.int64(0)

In [5]:
final_df.query('subfolder == "up2012_821" ')

,Unnamed: 0,constituency,party,age,profession,spouse_profession,n_crime_cases,assets,liabilities,education,subfolder,year
1789,0,AMETHI (CSJM NAGAR),SP,44,Property Delear,House wife,1,"Assets: Rs 1,83,59,616 ~1 Crore+",Liabilities: Nil,"Educational Details. Category: Graduate. B.A.,...",up2012_821,2012


In [6]:
ac_names_exp = final_df['constituency'].str.split("(", expand = True)
ac_names_exp.columns = ['name1', 'name2', 'name3', 'name4']
ac_names_exp['original'] = final_df['constituency'].copy()

# Initialize as object dtype so string values can be assigned (a np.nan
# init makes these float64, and newer pandas refuses to set strings into it)
ac_names_exp['ac_name'] = pd.Series(np.nan, index=ac_names_exp.index, dtype=object)
ac_names_exp['district'] = pd.Series(np.nan, index=ac_names_exp.index, dtype=object)
par3 = ac_names_exp['name3'].isnull()
ac_names_exp.loc[par3, 'ac_name'] = ac_names_exp.loc[par3, 'name1']
ac_names_exp.loc[par3, 'district'] = ac_names_exp.loc[par3, 'name2']

par4 = ac_names_exp['name4'].isnull() & ac_names_exp['name3'].notnull()
ac_names_exp.loc[par4, 'ac_name'] = ac_names_exp.loc[par4, 'name1'] + "(" + ac_names_exp.loc[par4, 'name2']
ac_names_exp.loc[par4, 'district'] = ac_names_exp.loc[par4, 'name3']

par5 = ac_names_exp.ac_name.isnull()
ac_names_exp.loc[par5, 'ac_name'] = ac_names_exp.loc[par5, 'name1'] + "(" + ac_names_exp.loc[par5, 'name2']
ac_names_exp.loc[par5, 'district'] = ac_names_exp.loc[par5, 'name3'] + "(" + ac_names_exp.loc[par5, 'name4']

ac_names_exp['district'] = ac_names_exp.district.str.replace(")", "")

In [7]:
# Extract candidate ID (number after the last "_")
final_df = pd.concat([final_df, ac_names_exp[['ac_name', 'district']]], axis = 1)
final_df["candidate_id"] = final_df["subfolder"].str.extract(r'_(\d+)$')

# Convert year and candidate_id to numeric type
final_df["year"] = pd.to_numeric(final_df["year"], errors='coerce')
final_df["candidate_id"] = pd.to_numeric(final_df["candidate_id"], errors='coerce')
final_df["state"] = final_df["subfolder"].str.extract(r'^([a-zA-Z]+)')

In [8]:
final_df["state"] = final_df["state"].replace("up","uttarpradesh" )
final_df["state"] = final_df["state"].replace("bihar","Bihar" )
final_df["state"] = final_df["state"].replace("bih","bihar" )
final_df["state"] = final_df["state"].replace("pb","punjab" )
final_df["state"] = final_df["state"].replace("haryana","Haryana" )
final_df["state"] = final_df["state"].replace("ha","Haryana" )
final_df["state"] = final_df["state"].replace("bihar", "Bihar" )

In [9]:
final_df.query('subfolder == "up2012_821" ')

,Unnamed: 0,constituency,party,age,profession,spouse_profession,n_crime_cases,assets,liabilities,education,subfolder,year,ac_name,district,candidate_id,state
1789,0,AMETHI (CSJM NAGAR),SP,44,Property Delear,House wife,1,"Assets: Rs 1,83,59,616 ~1 Crore+",Liabilities: Nil,"Educational Details. Category: Graduate. B.A.,...",up2012_821,2012,AMETHI,CSJM NAGAR,821,uttarpradesh


In [10]:
final_df.subfolder

0               bih2010_1019
1               bih2010_1030
2               bih2010_1036
3               bih2010_1048
4               bih2010_1060
                ...         
2620    uttarpradesh2022_953
2621     uttarpradesh2022_96
2622    uttarpradesh2022_960
2623     uttarpradesh2022_98
2624    uttarpradesh2022_997
Name: subfolder, Length: 2625, dtype: str

In [11]:
# Selecting the States
sel_states = ["uttarpradesh", "Bihar", "Haryana", "punjab"]
idx_st = final_df["state"].isin(sel_states)
winners_acs_sel = final_df.loc[ idx_st, : ].copy()
winners_acs_sel['state'] = winners_acs_sel.state.replace("uttarpradesh", 'UTTAR PRADESH')
winners_acs_sel['state'] = winners_acs_sel.state.replace("Bihar", 'BIHAR')
winners_acs_sel['state'] = winners_acs_sel.state.replace("Haryana", 'HARYANA')
winners_acs_sel['state'] = winners_acs_sel.state.replace("punjab", 'PUNJAB')
winners_acs_sel['state_old'] = winners_acs_sel['state'].copy()
winners_acs_sel['count'] = 1

In [12]:
winners_acs_sel['constituency'] = winners_acs_sel['constituency'].str.upper()
winners_acs_sel['constituency'] = winners_acs_sel['constituency'].str.replace(" )", ")")

In [13]:
winners_acs_sel['aux']=1

In [14]:
winners_acs_sel.groupby(['state', 'candidate_id'])['aux'].sum().reset_index()['aux'].value_counts()

aux
1    2241
2     171
3      14
Name: count, dtype: int64

In [15]:
winners_acs_sel['candidate_id'].value_counts()

candidate_id
95     9
64     7
152    6
45     6
75     6
      ..
939    1
943    1
953    1
960    1
997    1
Name: count, Length: 2029, dtype: int64

In [16]:
acs_myneta = pd.read_csv(fr'{int_path}/cross_walk_ac_myneta_clean.csv')
acs_myneta['const_name'] =  (acs_myneta['ASSEMBLY_1'] + " (" + acs_myneta['DISTRICT'] + ")").copy()

In [17]:
winner_merge = winners_acs_sel.merge(acs_myneta, how = 'left', indicator = True, 
                                    on = ['state', 'constituency'] )
col_winners = winners_acs_sel.columns

In [18]:
colsel = winners_acs_sel.columns
imperfect_merge = winner_merge.query('_merge == "left_only" ').loc[:, colsel].copy()
perfect_merge = winner_merge.query('_merge == "both" ').copy()
print(imperfect_merge.columns)
print(winners_acs_sel.columns)

Index(['Unnamed: 0', 'constituency', 'party', 'age', 'profession',
       'spouse_profession', 'n_crime_cases', 'assets', 'liabilities',
       'education', 'subfolder', 'year', 'ac_name', 'district', 'candidate_id',
       'state', 'state_old', 'count', 'aux'],
      dtype='str')
Index(['Unnamed: 0', 'constituency', 'party', 'age', 'profession',
       'spouse_profession', 'n_crime_cases', 'assets', 'liabilities',
       'education', 'subfolder', 'year', 'ac_name', 'district', 'candidate_id',
       'state', 'state_old', 'count', 'aux'],
      dtype='str')


In [19]:
acNmaes_id = perfect_merge[['acpost08ID', 'ac_uq_id']].drop_duplicates()

In [20]:
acs_myneta = pd.read_csv(fr'{int_path}/cross_walk_ac_myneta_clean.csv')
left80score = pd.read_csv(fr'{int_path}/names2check08_clean.csv')
left_need1_merge = pd.read_csv(fr'{int_path}/names2check_clean.csv') \
                        .query('score > 0.8')

left80score = left80score[['state_x', 'constituency_x', 'ac_uq_id']] \
                    .query('ac_uq_id.notnull()') \
                    .drop_duplicates()
left80score['source'] = 'left_only80'

left_need1_clean = left_need1_merge[['state_x', 'constituency_x', 'ac_uq_id']] \
                        .query('ac_uq_id.notnull()') \
                        .drop_duplicates()
left_need1_clean['source'] = 'left_only'
clean_inspection = pd.concat([left80score, left_need1_clean])
clean_inspection = clean_inspection.merge(acNmaes_id, how = 'left', indicator = True, on = 'ac_uq_id')
clean_perfect = clean_inspection.query('_merge == "both"').drop('_merge', axis = 1)


In [21]:
clean_perfect.columns=['state', 'constituency', 'ac_uq_id', 'source', 'acpost08ID']

In [22]:
acs = gpd.read_file(f'{int_path}/_0_2_3_ACs_right_shapefile.shp')

In [23]:
colsel = imperfect_merge.columns
imp_merge = imperfect_merge.merge(clean_perfect, on = ['state', 'constituency'], how = 'left', indicator = True)
imp_correct = imp_merge.query('_merge == "both" ').copy()
colsel = winners_acs_sel.columns
imp_incomplete = imp_merge.query('_merge == "left_only" ').copy().loc[:, colsel]
left_uq = imp_incomplete[['constituency', 'state']].drop_duplicates()


In [24]:
left_uq

,constituency,state
0,AURAI (MUZAFFARPUR),BIHAR
1,RAGHOPUR ( VAISHALI) (VAISHALI),BIHAR
2,SURYAGARHA (LAKHISARAI),BIHAR
3,RAGHOPUR (VAISHALI),BIHAR
160,BHOGNIPUR (RAMABAI NAGAR),UTTAR PRADESH
171,DHAULANA (PANCHSHEEL NAGAR),UTTAR PRADESH
214,AMETHI (CSJM NAGAR),UTTAR PRADESH


In [25]:
# Create mapping of constituency name to acpost08ID
constituency_to_id = {
    "SURYAGARHA (LAKHISARAI)": 645,
    "RAGHOPUR (VAISHALI)": 543,
    "RAGHOPUR ( VAISHALI) (VAISHALI)": 543,
    "AMETHI (CSJM NAGAR)": 3489,
    "AURAI (MUZAFFARPUR)": 707,
    "DHAULANA (PANCHSHEEL NAGAR)": 3554,
    "BHOGNIPUR (RAMABAI NAGAR)": 3459,
}

left_uq["acpost08ID"] = left_uq["constituency"].map(constituency_to_id)

# Check results
print(left_uq[["constituency", "acpost08ID"]].drop_duplicates().sort_values("acpost08ID"))

# Check for unmatched
unmatched = left_uq[left_uq["acpost08ID"].isna()]
print(f"\nUnmatched rows: {len(unmatched)}")


                        constituency  acpost08ID
1    RAGHOPUR ( VAISHALI) (VAISHALI)         543
3                RAGHOPUR (VAISHALI)         543
2            SURYAGARHA (LAKHISARAI)         645
0                AURAI (MUZAFFARPUR)         707
160        BHOGNIPUR (RAMABAI NAGAR)        3459
214              AMETHI (CSJM NAGAR)        3489
171      DHAULANA (PANCHSHEEL NAGAR)        3554

Unmatched rows: 0


In [26]:
imp_incomplete = imp_incomplete.merge(left_uq, on = ['constituency', 'state'])

In [27]:
colsel = [ 'subfolder', 'state', 'constituency', 'acpost08ID', 'year', 'age', 'profession', 
 'spouse_profession', 'n_crime_cases', 
 'assets', 'liabilities', 'education'  ]
clean_myneta_info = pd.concat([perfect_merge, imp_incomplete, imp_correct]).loc[:, colsel].drop_duplicates('subfolder')
clean_myneta_info.shape

(2625, 12)

In [28]:
clean_myneta_info.query('subfolder == "up2012_821" ')

,subfolder,state,constituency,acpost08ID,year,age,profession,spouse_profession,n_crime_cases,assets,liabilities,education
8,up2012_821,UTTAR PRADESH,AMETHI (CSJM NAGAR),3489.0,2012,44,Property Delear,House wife,1,"Assets: Rs 1,83,59,616 ~1 Crore+",Liabilities: Nil,"Educational Details. Category: Graduate. B.A.,..."


In [29]:
# Adding names of ACs
acs_raw = gpd.read_file(fr'{int_path}/acs_post2008_id.shp')
myneta_info = pd.read_csv(fr'{input_path}/my_neta/2008_onwards_winners_table.csv') \
                    .rename(columns = {'unique_id' : 'subfolder'})

In [30]:
gemini_path = (
    r"C:\Users\eunic\OneDrive\Documents\GitHub\ownpkg\myneta_llm"
    r"\data\gemini_self_prof_batch_historical\final"
    r"\2008_onwards_winners_table_gemini.csv"
)

# The final file has one row per politician.
gemini_prof = (
    pd.read_csv(
        gemini_path,
        usecols=["unique_id", "self_profession"],
    )
    .rename(
        columns={
            "unique_id": "subfolder",
            "self_profession": "self_profession_gemini",
        }
    )
)

# Convert the classification to nullable 0/1.
# Blank professions remain missing rather than becoming zero.
gemini_prof["self_profession_gemini"] = (
    gemini_prof["self_profession_gemini"]
    .replace({
        True: 1,
        False: 0,
        "True": 1,
        "False": 0,
    })
    .pipe(pd.to_numeric, errors="raise")
    .astype("Int64")
)

assert not myneta_info["subfolder"].duplicated().any()
assert not gemini_prof["subfolder"].duplicated().any()

# Check key overlap before constructing the final merge.
gemini_overlap = (
    myneta_info[["subfolder"]]
    .merge(
        gemini_prof[["subfolder"]],
        on="subfolder",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

overlap_counts = (
    gemini_overlap["_merge"]
    .value_counts()
    .reindex(["left_only", "right_only", "both"], fill_value=0)
)

print("Gemini merge-key overlap:")
print(overlap_counts)

# The final Gemini file was constructed for the complete 2,625-row table.
assert overlap_counts["left_only"] == 0, (
    f"{overlap_counts['left_only']} myneta observations are missing "
    "from the final Gemini dataset."
)
assert overlap_counts["right_only"] == 0, (
    f"{overlap_counts['right_only']} Gemini observations are missing "
    "from myneta_info."
)

myneta_info = myneta_info.merge(
    gemini_prof,
    on="subfolder",
    how="left",
    validate="one_to_one",
    indicator="_gemini_merge",
)

matched_gemini = myneta_info["_gemini_merge"].eq("both")


Gemini merge-key overlap:
_merge
left_only        0
right_only       0
both          2625
Name: count, dtype: int64


In [31]:
print(myneta_info[['self_profession', 'self_profession_gemini']].value_counts().reset_index())

   self_profession  self_profession_gemini  count
0            False                       0   1137
1             True                       1   1059
2             True                       0     48
3            False                       1     16


In [32]:

# Replace the old classification with the final Gemini classification.
# Missing Gemini classifications remain missing.
myneta_info["self_profession"] = (
    myneta_info["self_profession_gemini"]
    .astype("Float64")
)

print("Gemini matches:", int(matched_gemini.sum()))
print(
    "Gemini classified:",
    int(myneta_info["self_profession_gemini"].notna().sum()),
)
print(
    "Gemini missing profession:",
    int(myneta_info["self_profession_gemini"].isna().sum()),
)
print(
    "Final self_profession distribution:"
)
print(
    myneta_info["self_profession"]
    .value_counts(dropna=False)
    .sort_index()
)

myneta_info = myneta_info.drop(
    columns=["self_profession_gemini", "_gemini_merge"]
)

Gemini matches: 2625
Gemini classified: 2260
Gemini missing profession: 365
Final self_profession distribution:
self_profession
0.0     1185
1.0     1075
<NA>     365
Name: count, dtype: Int64


In [33]:
myneta_info.columns

Index(['Unnamed: 0', 'subfolder', 'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'spouse_profession', 'name', 'ac_name', 'state', 'year', 'education'],
      dtype='str')

In [34]:
myneta_acnames = clean_myneta_info.merge(acs_raw, on ='acpost08ID', how = 'left', indicator = True).drop('geometry', axis = 1)
print(myneta_acnames._merge.value_counts())
myneta_acnames = myneta_acnames.drop('_merge', axis = 1)
myneta_final = myneta_acnames.merge(myneta_info, on = 'subfolder', indicator = True, how = 'left')
print(myneta_final._merge.value_counts())
myneta_final = myneta_final.drop('_merge', axis = 1)


_merge
both          2625
left_only        0
right_only       0
Name: count, dtype: int64
_merge
both          2625
left_only        0
right_only       0
Name: count, dtype: int64


In [35]:
myneta_final.columns

Index(['subfolder', 'state_x', 'constituency', 'acpost08ID', 'year_x', 'age',
       'profession', 'spouse_profession_x', 'n_crime_cases', 'assets',
       'liabilities', 'education_x', 'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT',
       'PARLIAMENT', 'P_NAME', 'STATE_UT', 'Unnamed: 0',
       'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'spouse_profession_y', 'name', 'ac_name', 'state_y', 'year_y',
       'education_y'],
      dtype='str')

In [36]:
colsel = ['state_x', 'constituency', 'acpost08ID', 'year_x', 'age',
'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT',
'P_NAME', 'STATE_UT', 'dependent_1_owns_agricultural_assets',
'dependent_2_owns_agricultural_assets',
'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
'spouse_owns_agricultural_assets', 'self_profession', 'profession', 'spouse_profession_x',
'spouse_profession_y', 'education_y', 'subfolder']

In [37]:
myneta_final.isna().sum()

subfolder                                 0
state_x                                   0
constituency                              0
acpost08ID                                0
year_x                                    0
age                                       0
profession                              365
spouse_profession_x                     555
n_crime_cases                             0
assets                                    0
liabilities                               0
education_x                               0
ASSEMBLY                                  0
ASSEMBLY_1                                0
DISTRICT                                  0
PARLIAMENT                                0
P_NAME                                    0
STATE_UT                                  0
Unnamed: 0                                0
dependent_1_owns_agricultural_assets      0
dependent_2_owns_agricultural_assets      0
dependent_3_owns_agricultural_assets      0
self_owns_agricultural_assets   

In [38]:
clean_data = myneta_final.loc[:, colsel].copy()

In [39]:
clean_data.columns = ['state', 'constituency', 'acpost08ID', 'year', 'age',
'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT',
'P_NAME', 'STATE_UT', 'dependent_1_owns_agricultural_assets',
'dependent_2_owns_agricultural_assets',
'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
'spouse_owns_agricultural_assets', 'self_profession', 'self_profession_raw', 'spouse_profession_raw',
'spouse_profession', 'education', 'subfolder']


In [40]:
clean_data['unique_id'] = clean_data.subfolder

In [41]:
clean_data.query('subfolder == "up2012_821" ')

,state,constituency,acpost08ID,year,age,ASSEMBLY,ASSEMBLY_1,DISTRICT,PARLIAMENT,P_NAME,...,dependent_3_owns_agricultural_assets,self_owns_agricultural_assets,spouse_owns_agricultural_assets,self_profession,self_profession_raw,spouse_profession_raw,spouse_profession,education,subfolder,unique_id
2350,UTTAR PRADESH,AMETHI (CSJM NAGAR),3489.0,2012,44,186,AMETHI,SULTANPUR,37.0,AMETHI,...,False,True,False,0.0,Property Delear,House wife,False,Graduate,up2012_821,up2012_821


In [42]:
clean_acs = gpd.read_file(fr"{int_path}/_0_2_3_ACs_right_shapefile.shp")

In [43]:
final_df = clean_data.merge(clean_acs, how = 'left', on = ['STATE_UT', 'DISTRICT', 'ASSEMBLY_1', 'ASSEMBLY'], 
                 indicator = True).drop('geometry', axis = 1)

In [44]:
final_df._merge.value_counts()

_merge
both          2625
left_only        0
right_only       0
Name: count, dtype: int64

In [45]:
final_df.query('subfolder == "up2012_821" ')

,state,constituency,acpost08ID_x,year,age,ASSEMBLY,ASSEMBLY_1,DISTRICT,PARLIAMENT,P_NAME,...,self_profession,self_profession_raw,spouse_profession_raw,spouse_profession,education,subfolder,unique_id,ac_uq_id,acpost08ID_y,_merge
2350,UTTAR PRADESH,AMETHI (CSJM NAGAR),3489.0,2012,44,186,AMETHI,SULTANPUR,37.0,AMETHI,...,0.0,Property Delear,House wife,False,Graduate,up2012_821,up2012_821,831,3489.0,both


In [46]:
final_df.columns

Index(['state', 'constituency', 'acpost08ID_x', 'year', 'age', 'ASSEMBLY',
       'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT',
       'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'self_profession_raw', 'spouse_profession_raw', 'spouse_profession',
       'education', 'subfolder', 'unique_id', 'ac_uq_id', 'acpost08ID_y',
       '_merge'],
      dtype='str')

In [47]:
bool_cols = ["dependent_1_owns_agricultural_assets",
             "dependent_2_owns_agricultural_assets",
             "dependent_3_owns_agricultural_assets",
             "self_owns_agricultural_assets",
             "spouse_owns_agricultural_assets",
             "self_profession",
             "spouse_profession"]

final_df[bool_cols] = final_df[bool_cols].astype(float)

In [48]:
# Cast to float so np.nan can be stored (these cols were cast to int above,
# and newer pandas refuses to set NaN into an int64 column)
final_df['self_profession'] = final_df['self_profession'].astype('float')
final_df['spouse_profession'] = final_df['spouse_profession'].astype('float')
final_df.loc[ final_df.self_profession_raw.isna(), 'self_profession']= np.nan
final_df.loc[ final_df.spouse_profession_raw.isna(), 'spouse_profession']= np.nan

In [49]:
final_df.isna().sum()

state                                     0
constituency                              0
acpost08ID_x                              0
year                                      0
age                                       0
ASSEMBLY                                  0
ASSEMBLY_1                                0
DISTRICT                                  0
PARLIAMENT                                0
P_NAME                                    0
STATE_UT                                  0
dependent_1_owns_agricultural_assets      0
dependent_2_owns_agricultural_assets      0
dependent_3_owns_agricultural_assets      0
self_owns_agricultural_assets             0
spouse_owns_agricultural_assets           0
self_profession                         365
self_profession_raw                     365
spouse_profession_raw                   555
spouse_profession                       555
education                                 0
subfolder                                 0
unique_id                       

In [50]:
final_df.columns

Index(['state', 'constituency', 'acpost08ID_x', 'year', 'age', 'ASSEMBLY',
       'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT',
       'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'self_profession_raw', 'spouse_profession_raw', 'spouse_profession',
       'education', 'subfolder', 'unique_id', 'ac_uq_id', 'acpost08ID_y',
       '_merge'],
      dtype='str')

In [51]:
colsel = ['state', 'constituency', 'acpost08ID', 'election_year', 'age', 'ASSEMBLY', 'ASSEMBLY_1',
       'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT',
       'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'self_profession_raw', 'spouse_profession_raw', 'spouse_profession',
       'education', 'subfolder', 'unique_id', 'ac_uq_id', 'acpost08ID_y', '_merge']
final_df.columns = colsel

In [52]:
colsel = ['state', 'ac_uq_id', 'constituency', 'acpost08ID', 'subfolder', 'unique_id', 'election_year', 'age', 'ASSEMBLY', 'ASSEMBLY_1',
       'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT',
       'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'self_profession_raw', 'spouse_profession_raw', 'spouse_profession',
       'education']

In [53]:
final_df = final_df.loc[:, colsel].copy()

In [55]:
# ---- Add winner vote share & total votes -------------------------------------
# Source: IV_AC_candidates.csv (raw IndiaVotes/MyNeta results); winner = Position==1.
#   vote_share  = winner's share of valid votes (%)   (from "Votes %")
#   total_votes = winner's raw vote count             (from "Votes")
# AC names are spelled inconsistently across sources, so a plain constituency merge
# only hits ~68%. Use a safe cascade that only ever uses UNAMBIGUOUS keys
# (drop_duplicates(..., keep=False)) so colliding AC names never mis-assign.
import re

def _norm_name(s):
    s = str(s).upper()
    s = re.sub(r'\bALIAS\b.*', '', s)      # drop "ALIAS ..." / "@ ..." tails
    s = re.sub(r'[^A-Z ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

# --- winner-level vote table ---
votes = pd.read_csv(fr"{input_path}/my_neta/IV_AC_candidates.csv")
w = votes[votes["Position"] == 1].copy()
w["state"]         = w["Election"].str.rsplit(" ", n=1).str[0].str.upper().str.strip()
w["election_year"] = w["Election"].str.rsplit(" ", n=1).str[1].astype(int)
w["vote_share"]    = w["Votes %"].str.replace("%", "", regex=False).astype(float)
w["total_votes"]   = pd.to_numeric(w["Votes"], errors="coerce")
w["cons"] = (w["AC_name"].str.upper().str.strip() + " (" + w["District"].str.upper().str.strip() + ")")
w["acn"]  = w["AC_name"].str.upper().str.strip()
w["nk"]   = w["Name"].map(_norm_name)

# --- panel-side keys (bring winner name in from the winners table via subfolder) ---
_names = pd.read_csv(fr"{input_path}/my_neta/2008_onwards_winners_table.csv") \
           .rename(columns={"unique_id": "subfolder"})[["subfolder", "name"]]
final_df = final_df.merge(_names, on="subfolder", how="left")
final_df["cons"] = final_df["constituency"].str.upper().str.strip()
final_df["acn"]  = final_df["constituency"].str.split(r"\s*\(").str[0].str.upper().str.strip()
final_df["nk"]   = final_df["name"].map(_norm_name)

# --- safe cascade: first match wins, unambiguous keys only ---
final_df["vote_share"]  = np.nan
final_df["total_votes"] = np.nan
final_df["_vote_src"]   = ""

def _fill(keys, label):
    lk = w.drop_duplicates(keys, keep=False).set_index(keys)[["vote_share", "total_votes"]]
    todo = final_df[final_df["vote_share"].isna()]
    idx = todo.set_index(keys).index
    hit = idx.isin(lk.index)
    matched_pos = todo.index[hit]
    vals = lk.loc[idx[hit]]
    final_df.loc[matched_pos, "vote_share"]  = vals["vote_share"].values
    final_df.loc[matched_pos, "total_votes"] = vals["total_votes"].values
    final_df.loc[matched_pos, "_vote_src"]   = label

_fill(["state", "election_year", "cons"], "constituency")
_fill(["state", "election_year", "acn"],  "acname")
_fill(["state", "election_year", "nk"],   "name")

# --- diagnostics ---
print("Vote match by source:")
print(final_df["_vote_src"].replace("", "UNMATCHED").value_counts().to_string())
n_unmatched = final_df["vote_share"].isna().sum()
print(f"\nMatched {len(final_df) - n_unmatched}/{len(final_df)}  |  unmatched: {n_unmatched}")
if n_unmatched:
    print("\nUnmatched winners (left as NaN):")
    print(final_df.loc[final_df["vote_share"].isna(),
                       ["state", "election_year", "constituency", "name"]].to_string())

# --- drop helper columns; keep only the two new variables on final_df ---
final_df = final_df.drop(columns=["name", "cons", "acn", "nk", "_vote_src"])


Vote match by source:
_vote_src
constituency    1781
acname           737
name              79
UNMATCHED         28

Matched 2597/2625  |  unmatched: 28

Unmatched winners (left as NaN):
              state  election_year                     constituency                           name
4             BIHAR           2010               GORIAKOTHI (SIWAN)         Bhumendra Narayn Singh
38            BIHAR           2010         GAURA BAURAM (DARBHANGA)                    IZHAR AHMAD
127           BIHAR           2010        TRIVENIGANJ (SC) (SUPAUL)                     Amala Devi
139           BIHAR           2010                BRAHAMPUR (BUXAR)  DIL MARNI DEVI @ DILMANI DEVI
204           BIHAR           2010           BHORE (SC) (GOPALGANJ)                INDRADEO MANJHI
211           BIHAR           2010      KALYANPUR (SC) (SAMASTIPUR)                Ramsevak Hazari
715         HARYANA           2009           AMBALA CANTT. (AMBALA)                       Anil Viz
1277  UTTAR PRADESH  

In [56]:
# First, let's see what the differences look like
print("Unique 'state' values:")
print(sorted(final_df["state"].unique()))
print("\nUnique 'STATE_UT' values:")
print(sorted(final_df["STATE_UT"].unique()))

# Clean both columns: strip whitespace, uppercase, remove extra spaces
final_df["state_clean"] = final_df["state"].str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
final_df["STATE_UT_clean"] = final_df["STATE_UT"].str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

# Check mismatches after cleaning
mismatch = final_df[final_df["state_clean"] != final_df["STATE_UT_clean"]]
print(f"\nMismatched rows after cleaning: {len(mismatch)}")

if len(mismatch) > 0:
    print("\nRemaining mismatches:")
    print(mismatch[["state_clean", "STATE_UT_clean"]].drop_duplicates())

Unique 'state' values:
['BIHAR', 'HARYANA', 'PUNJAB', 'UTTAR PRADESH']

Unique 'STATE_UT' values:
['BIHAR', 'HARYANA', 'PUNJAB', 'UTTAR PRADESH']

Mismatched rows after cleaning: 0


In [57]:
final_df.to_csv(fr'{int_path}/_ac_covs_myneta.csv', index = False)

In [58]:
df1 = pd.read_csv(fr'{int_path}/_ac_covs_myneta.csv')

In [59]:
np.sort(np.array(df1.ac_uq_id.unique().tolist()))

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

## Generation of Panel Data

In [60]:
# Importing Information
final_df = pd.read_csv(fr'{int_path}/_ac_covs_myneta.csv')
election_info = pd.read_csv(fr'{input_path}/my_neta/ac_india_elec_yr_clean.csv').rename(columns = {'year' : 'election_year'})

#Year and month data generation
months = pd.DataFrame({'month' : np.arange(1, 13)})
months['count']  = 1
yrs = pd.DataFrame({'year' : np.arange(2000, 2030)})
yrs['count'] = 1
months = months.merge(yrs).sort_values(['year', 'month'])

# ACs 
ac_by_year = final_df[['state', 'ac_uq_id']].drop_duplicates()
ac_by_year['count']=1
ac_by_year = ac_by_year.merge(months).drop('count', axis = 1)

ac_panel = ac_by_year.merge( election_info, left_on = ['state', 'year', 'month'], 
                 right_on = [ 'state', 'year_take', 'month_take'], how = 'left')
ac_panel = ac_panel.sort_values(['state', 'ac_uq_id', 'year', 'month'])
ac_panel[ac_panel.columns.difference(['state', 'ac_uq_id'])] = (
    ac_panel.groupby(['state', 'ac_uq_id'])[ac_panel.columns.difference(['state', 'ac_uq_id'])].ffill()
)      
# Create a numeric year-month for easy comparison
ac_panel["ym"] = ac_panel["year"] * 12 + ac_panel["month"]
ac_panel["ym_take"] = ac_panel["year_take"] * 12 + ac_panel["month_take"]

# Keep rows where year-month >= year_take/month_take AND within 60 months
ac_panel = ac_panel[
    (ac_panel["ym"] >= ac_panel["ym_take"]) & 
    (ac_panel["ym"] < ac_panel["ym_take"] + 60)
]
# Clean up helper columns
ac_panel = ac_panel.drop(columns=["ym", "ym_take"])
ac_panel = ac_panel.drop(['count', 'day_take', 'day_counting', 'month_counting', 'year_counting', 'nseats'], axis = 1)
ac_panel["ym"] = ac_panel["year"] * 12 + ac_panel["month"]
ac_panel["ym_take"] = ac_panel["year_take"] * 12 + ac_panel["month_take"]
ac_panel["yeargov"] = ((ac_panel["ym"] - ac_panel["ym_take"]) // 12) + 1

In [61]:

panel_acs_elec_covs = ac_panel.merge(final_df, on = ['state', 'ac_uq_id', 'election_year'], how = 'left')
fixed_cols = ['acpost08ID', 'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT', 'P_NAME', 'STATE_UT']
# Fill AC-identifying attributes WITHIN each AC (they are constant per ac_uq_id),
# not globally. Filling with a single global value would stamp one AC's identity
# onto every AC that had a missing election-year match.
panel_acs_elec_covs[fixed_cols] = (
    panel_acs_elec_covs
        .groupby('ac_uq_id')[fixed_cols]
        .transform(lambda s: s.ffill().bfill())
)
# Filter based on the condition: start <= current < end
panel_acs_elec_covs.to_stata(fr'{int_path}/panel_data_election_year.dta')


C:\Users\eunic\AppData\Local\Temp\ipykernel_25664\264882006.py:12: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    dependent_1_owns_agricultural_assets   ->   dependent_1_owns_agricultural_as
    dependent_2_owns_agricultural_assets   ->   dependent_2_owns_agricultural_as
    dependent_3_owns_agricultural_assets   ->   dependent_3_owns_agricultural_as

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  panel_acs_elec_covs.to_stata(fr'{int_path}/panel_data_election_year.dta')


In [62]:
panel_acs_elec_covs.to_parquet(fr'{int_path}/panel_data_election_year.parquet')